In [16]:
import pandas as pd

df = pd.read_csv('data/processed/LA_weather_aqi.csv', parse_dates=['Date Local'])

print(df.shape)
print(df.dtypes)
print(df.head())

(1796, 14)
Date Local                    datetime64[ns]
pm25_aqi                             float64
ozone_aqi                            float64
AQI                                  float64
temperature_2m_max                   float64
temperature_2m_min                   float64
precipitation_sum                    float64
windspeed_10m_max                    float64
winddirection_10m_dominant             int64
dewpoint_2m_mean                     float64
relative_humidity_2m_mean              int64
pressure_msl_mean                    float64
cloudcover_mean                        int64
shortwave_radiation_sum              float64
dtype: object
  Date Local  pm25_aqi  ozone_aqi   AQI  temperature_2m_max  \
0 2020-01-01      66.0       34.0  66.0                17.8   
1 2020-01-02      66.0       31.0  66.0                17.1   
2 2020-01-03      57.0       21.0  57.0                20.9   
3 2020-01-04      72.0       33.0  72.0                19.0   
4 2020-01-05      62.0       2

In [17]:
df = df.sort_values(by='Date Local').reset_index(drop=True)


In [18]:
df['month'] = df['Date Local'].dt.month
df['day_of_week'] = df['Date Local'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [19]:
print(df[['Date Local', 'month', 'day_of_week', 'is_weekend']].head(10))

  Date Local  month  day_of_week  is_weekend
0 2020-01-01      1            2           0
1 2020-01-02      1            3           0
2 2020-01-03      1            4           0
3 2020-01-04      1            5           1
4 2020-01-05      1            6           1
5 2020-01-06      1            0           0
6 2020-01-07      1            1           0
7 2020-01-08      1            2           0
8 2020-01-10      1            4           0
9 2020-01-11      1            5           1


In [20]:
df['aqi_lag_1'] = df['AQI'].shift(1)
df['aqi_lag_2'] = df['AQI'].shift(2)
df['aqi_lag_3'] = df['AQI'].shift(3)
df['aqi_lag_7'] = df['AQI'].shift(7)
df['aqi_lag_14'] = df['AQI'].shift(14)

In [21]:
print(df[['Date Local', 'AQI', 'aqi_lag_1', 'aqi_lag_7']].head(10))

  Date Local   AQI  aqi_lag_1  aqi_lag_7
0 2020-01-01  66.0        NaN        NaN
1 2020-01-02  66.0       66.0        NaN
2 2020-01-03  57.0       66.0        NaN
3 2020-01-04  72.0       57.0        NaN
4 2020-01-05  62.0       72.0        NaN
5 2020-01-06  42.0       62.0        NaN
6 2020-01-07  36.0       42.0        NaN
7 2020-01-08  60.0       36.0       66.0
8 2020-01-10  57.0       60.0       66.0
9 2020-01-11  65.0       57.0       57.0


In [22]:
df['ozone_dominant'] = (df['ozone_aqi'] > df['pm25_aqi']).astype(int)
df['ozone_dominant_lag1'] = df['ozone_dominant'].shift(1)

In [23]:
df['had_rain'] = (df['precipitation_sum'] >= 2.5).astype(int)

# Compute days since last rain event
days_since_rain = []
last_rain_index = -1

for i, had_rain in enumerate(df['had_rain']):
    if had_rain:
        days_since_rain.append(0)
        last_rain_index = i
    else:
        if last_rain_index == -1:
            # No rain so far
            days_since_rain.append(None)
        else:
            days_since_rain.append(i - last_rain_index)

df['days_since_rain'] = days_since_rain

In [24]:
print(df[['Date Local', 'precipitation_sum', 'had_rain', 'days_since_rain']].head(20))

   Date Local  precipitation_sum  had_rain  days_since_rain
0  2020-01-01                0.0         0              NaN
1  2020-01-02                0.0         0              NaN
2  2020-01-03                0.0         0              NaN
3  2020-01-04                0.0         0              NaN
4  2020-01-05                0.0         0              NaN
5  2020-01-06                0.0         0              NaN
6  2020-01-07                0.0         0              NaN
7  2020-01-08                0.0         0              NaN
8  2020-01-10                0.0         0              NaN
9  2020-01-11                0.0         0              NaN
10 2020-01-12                0.0         0              NaN
11 2020-01-13                0.0         0              NaN
12 2020-01-14                0.0         0              NaN
13 2020-01-15                0.0         0              NaN
14 2020-01-16                2.6         1              0.0
15 2020-01-17                4.6        

In [25]:
df['aqi_rolling_7d_mean'] = df['AQI'].shift(1).rolling(7).mean()
df['aqi_rolling_7d_std'] = df['AQI'].shift(1).rolling(7).std()

In [26]:
print(df[['Date Local', 'AQI', 'aqi_rolling_7d_mean', 'aqi_rolling_7d_std']].head(10))
print(df.isnull().sum())

  Date Local   AQI  aqi_rolling_7d_mean  aqi_rolling_7d_std
0 2020-01-01  66.0                  NaN                 NaN
1 2020-01-02  66.0                  NaN                 NaN
2 2020-01-03  57.0                  NaN                 NaN
3 2020-01-04  72.0                  NaN                 NaN
4 2020-01-05  62.0                  NaN                 NaN
5 2020-01-06  42.0                  NaN                 NaN
6 2020-01-07  36.0                  NaN                 NaN
7 2020-01-08  60.0            57.285714           13.400426
8 2020-01-10  57.0            56.428571           12.933898
9 2020-01-11  65.0            55.142857           12.253279
Date Local                     0
pm25_aqi                       0
ozone_aqi                      0
AQI                            0
temperature_2m_max             0
temperature_2m_min             0
precipitation_sum              0
windspeed_10m_max              0
winddirection_10m_dominant     0
dewpoint_2m_mean               0
relative_h

In [27]:
print(df.isnull().sum())
print(f"Total rows before dropping NaNs: {len(df)}")

df = df.dropna()

print(f"Total rows after dropping NaNs: {len(df)}")

Date Local                     0
pm25_aqi                       0
ozone_aqi                      0
AQI                            0
temperature_2m_max             0
temperature_2m_min             0
precipitation_sum              0
windspeed_10m_max              0
winddirection_10m_dominant     0
dewpoint_2m_mean               0
relative_humidity_2m_mean      0
pressure_msl_mean              0
cloudcover_mean                0
shortwave_radiation_sum        0
month                          0
day_of_week                    0
is_weekend                     0
aqi_lag_1                      1
aqi_lag_2                      2
aqi_lag_3                      3
aqi_lag_7                      7
aqi_lag_14                    14
ozone_dominant                 0
ozone_dominant_lag1            1
had_rain                       0
days_since_rain               14
aqi_rolling_7d_mean            7
aqi_rolling_7d_std             7
dtype: int64
Total rows before dropping NaNs: 1796
Total rows after dropping

In [28]:
df_spring = df[df['month'].isin([3, 4, 5])]

print(df_spring.shape)
print(df_spring['month'].value_counts())

(455, 28)
month
5    154
3    152
4    149
Name: count, dtype: int64


In [29]:
df_spring.to_csv('data/processed/la_features.csv', index=False)
print(df_spring.columns.tolist())

['Date Local', 'pm25_aqi', 'ozone_aqi', 'AQI', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'winddirection_10m_dominant', 'dewpoint_2m_mean', 'relative_humidity_2m_mean', 'pressure_msl_mean', 'cloudcover_mean', 'shortwave_radiation_sum', 'month', 'day_of_week', 'is_weekend', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_7', 'aqi_lag_14', 'ozone_dominant', 'ozone_dominant_lag1', 'had_rain', 'days_since_rain', 'aqi_rolling_7d_mean', 'aqi_rolling_7d_std']


In [30]:
features_to_drop = ['Date Local', 'had_rain', 'pm25_aqi', 'ozone_aqi', 'ozone_dominant']

X = df_spring.drop(columns=features_to_drop + ['AQI'])
Y = df_spring['AQI']

print(X.shape)
print(X.columns.tolist())

(455, 22)
['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'winddirection_10m_dominant', 'dewpoint_2m_mean', 'relative_humidity_2m_mean', 'pressure_msl_mean', 'cloudcover_mean', 'shortwave_radiation_sum', 'month', 'day_of_week', 'is_weekend', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_7', 'aqi_lag_14', 'ozone_dominant_lag1', 'days_since_rain', 'aqi_rolling_7d_mean', 'aqi_rolling_7d_std']
